In [1]:
import pandas as pd

# Load TrustMesh economic data
df = pd.read_csv("../data/trustmesh_economic_data.csv")
df["month"] = pd.to_datetime(df["month"])

# Use the latest economic state of each business
profiles = (
    df.sort_values("month")
      .groupby("business_id")
      .tail(1)
      .reset_index(drop=True)
)

print("Business profiles:", profiles.shape)
print("Businesses:", profiles["business_id"].nunique())

Business profiles: (500, 32)
Businesses: 500


In [2]:
def generate_recommendation(row):
    recommendations = []

    if row["cash_flow_margin_pct"] < 20:
        recommendations.append("Improve cash flow management")

    if row["expense_burden_pct"] > 60:
        recommendations.append("Reduce operating expenses")

    if row["payment_discipline_pct"] < 70:
        recommendations.append("Improve payment discipline")

    if row["supplier_reliability_pct"] < 70:
        recommendations.append("Strengthen supplier reliability")

    if row["demand_stability_pct"] < 70:
        recommendations.append("Stabilize customer demand")

    if row["business_continuity_pct"] < 70:
        recommendations.append("Strengthen business continuity")

    if row["resilience_pct"] < 70:
        recommendations.append("Build financial resilience")

    if row["economic_reputation_index"] >= 75 and not recommendations:
        recommendations.append("Maintain current economic practices")
    elif not recommendations:
        recommendations.append("Continue monitoring economic performance")

    return recommendations


profiles["recommendations"] = profiles.apply(generate_recommendation, axis=1)

# Convert recommendations into a readable format
profiles["recommendation_count"] = profiles["recommendations"].str.len()
profiles["recommendations"] = profiles["recommendations"].apply(
    lambda x: "; ".join(x)
)

print("Businesses with recommendations:", len(profiles))
print("Average recommendations per business:", 
      profiles["recommendation_count"].mean().round(2))

profiles[[
    "business_id",
    "economic_reputation_index",
    "recommendation_count",
    "recommendations"
]].head(10)

Businesses with recommendations: 500
Average recommendations per business: 1.55


,business_id,economic_reputation_index,recommendation_count,recommendations
0,B490,72.30,1,Continue monitoring economic performance
1,B481,55.48,4,Improve cash flow management; Reduce operating...
2,B486,61.81,2,Improve cash flow management; Reduce operating...
3,B497,67.35,1,Continue monitoring economic performance
4,B493,66.40,2,Strengthen supplier reliability; Stabilize cus...
5,B495,63.82,1,Continue monitoring economic performance
6,B484,70.02,2,Reduce operating expenses; Strengthen supplier...
7,B491,66.87,1,Continue monitoring economic performance
8,B485,63.59,1,Improve payment discipline
9,B499,68.45,1,Stabilize customer demand


In [3]:
def assign_priority(row):
    if row["economic_reputation_index"] < 50 or row["recommendation_count"] >= 4:
        return "High"
    elif row["economic_reputation_index"] < 65 or row["recommendation_count"] >= 2:
        return "Medium"
    return "Low"


profiles["recommendation_priority"] = profiles.apply(assign_priority, axis=1)

print("Recommendation priority:")
print(profiles["recommendation_priority"].value_counts())

print("\nSample recommendations:")
print(
    profiles[
        [
            "business_id",
            "economic_reputation_index",
            "recommendation_priority",
            "recommendations"
        ]
    ].head(10).to_string(index=False)
)

Recommendation priority:
recommendation_priority
Low       284
Medium    192
High       24
Name: count, dtype: int64

Sample recommendations:
business_id  economic_reputation_index recommendation_priority                                                                                                recommendations
       B490                      72.30                     Low                                                                       Continue monitoring economic performance
       B481                      55.48                    High Improve cash flow management; Reduce operating expenses; Improve payment discipline; Stabilize customer demand
       B486                      61.81                  Medium                                                        Improve cash flow management; Reduce operating expenses
       B497                      67.35                     Low                                                                       Continue monitoring economic 

In [4]:
# Prepare final recommendation output
recommendation_output = profiles[
    [
        "business_id",
        "owner_id",
        "business_type",
        "environment_type",
        "location",
        "economic_reputation_index",
        "recommendation_priority",
        "recommendation_count",
        "recommendations"
    ]
].copy()

# Save results
recommendation_output.to_csv(
    "../data/business_recommendations.csv",
    index=False
)

print("Recommendation output saved.")
print("Shape:", recommendation_output.shape)
print("\nPriority distribution:")
print(recommendation_output["recommendation_priority"].value_counts())

print("\nRecommendation output preview:")
print(recommendation_output.head(10).to_string(index=False))

Recommendation output saved.
Shape: (500, 9)

Priority distribution:
recommendation_priority
Low       284
Medium    192
High       24
Name: count, dtype: int64

Recommendation output preview:
business_id owner_id          business_type environment_type  location  economic_reputation_index recommendation_priority  recommendation_count                                                                                                recommendations
       B490     O490 Micro Service Business       Semi-Urban Ahmedabad                      72.30                     Low                     1                                                                       Continue monitoring economic performance
       B481     O481            Food Vendor       Semi-Urban    Jaipur                      55.48                    High                     4 Improve cash flow management; Reduce operating expenses; Improve payment discipline; Stabilize customer demand
       B486     O486            Retail Sho

## Recommendation Engine Findings

The Recommendation Engine translates business-level economic signals into actionable recommendations.

Recommendations are generated using key indicators including cash-flow margin, expense burden, payment discipline, supplier reliability, demand stability, business continuity, and resilience.

Each business is assigned a recommendation priority based on its Economic Reputation Index and the number of identified improvement areas.

The final output contains **500 business-level recommendation profiles**:

- **284 Low priority**
- **192 Medium priority**
- **24 High priority**

The recommendation output is saved as `data/business_recommendations.csv`.

This rule-based recommendation layer provides an interpretable decision-support mechanism that can later be integrated with the Economic Knowledge Graph, Business Twin Simulator, Explainable AI layer, and application dashboard.